## Nhúng (Embeddings)

Trong ví dụ trước đây, chúng ta đã thao tác trên các vectơ túi từ có chiều cao với độ dài `vocab_size`, và chúng ta đã rõ ràng chuyển đổi từ các vectơ biểu diễn vị trí có chiều thấp sang biểu diễn one-hot thưa thớt. Biểu diễn one-hot này không hiệu quả về bộ nhớ, thêm vào đó, mỗi từ được xử lý độc lập với nhau, nghĩa là các vectơ mã hóa one-hot không thể hiện bất kỳ sự tương đồng ngữ nghĩa nào giữa các từ.

Trong phần này, chúng ta sẽ tiếp tục khám phá bộ dữ liệu **News AG**. Để bắt đầu, hãy tải dữ liệu và lấy một số định nghĩa từ sổ tay trước.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## What is embedding?

The idea of **embedding** is to represent words by lower-dimensional dense vectors, which somehow reflect semantic meaning of a word. We will later discuss how to build meaningful word embeddings, but for now let's just think of embeddings as a way to lower dimensionality of a word vector. 

So, embedding layer would take a word as an input, and produce an output vector of specified `embedding_size`. In a sense, it is very similar to `Linear` layer, but instead of taking one-hot encoded vector, it will be able to take a word number as an input.

By using embedding layer as a first layer in our network, we can switch from bag-of-words to **embedding bag** model, where we first convert each word in our text into corresponding embedding, and then compute some aggregate function over all those embeddings, such as `sum`, `average` or `max`.  

![Image showing an embedding classifier for five sequence words.](../../../../../translated_images/vi/embedding-classifier-example.b77f021a7ee67eee.webp)

Our classifier neural network will start with embedding layer, then aggregation layer, and linear classifier on top of it:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Xử lý kích thước chuỗi biến đổi

Là kết quả của kiến trúc này, các minibatch gửi vào mạng cần được tạo theo một cách nhất định. Trong phần trước, khi sử dụng bag-of-words, tất cả các tensor BoW trong một minibatch có kích thước bằng nhau `vocab_size`, bất kể độ dài thực tế của chuỗi văn bản. Khi chuyển sang word embeddings, chúng ta sẽ có số lượng từ biến đổi trong mỗi mẫu văn bản, và khi kết hợp các mẫu thành minibatch, chúng ta cần áp dụng một số phần padding.

Việc này có thể thực hiện bằng cách sử dụng kỹ thuật cung cấp một hàm `collate_fn` cho datasource:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Đào tạo bộ phân loại embedding

Bây giờ chúng ta đã định nghĩa dataloader phù hợp, chúng ta có thể đào tạo mô hình sử dụng hàm đào tạo mà chúng ta đã định nghĩa trong đơn vị trước:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Lưu ý**: Chúng ta chỉ đào tạo với 25k bản ghi ở đây (ít hơn một epoch đầy đủ) vì lý do thời gian, nhưng bạn có thể tiếp tục đào tạo, viết một hàm để đào tạo trong nhiều epoch, và thử nghiệm với tham số tốc độ học để đạt được độ chính xác cao hơn. Bạn nên có thể đạt đến độ chính xác khoảng 90%.


### Lớp EmbeddingBag và Biểu diễn Dãy có Độ dài Biến đổi

Trong kiến trúc trước, chúng ta cần phải điền đủ (pad) tất cả các dãy về cùng một độ dài để có thể đặt chúng vào một minibatch. Đây không phải là cách hiệu quả nhất để biểu diễn các dãy có độ dài biến đổi - một cách tiếp cận khác là sử dụng vector **offset**, nó sẽ giữ các offset của tất cả các dãy được lưu trong một vector lớn.

![Ảnh minh họa biểu diễn dãy dưới dạng offset](../../../../../translated_images/vi/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Lưu ý**: Trong hình trên, chúng tôi hiển thị một dãy ký tự, nhưng trong ví dụ của chúng ta đang làm việc với các dãy từ. Tuy nhiên, nguyên lý chung của việc biểu diễn các dãy với vector offset vẫn giữ nguyên.

Để làm việc với biểu diễn offset, chúng ta sử dụng lớp [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). Nó tương tự như `Embedding`, nhưng nó nhận vector nội dung và vector offset làm đầu vào, và nó cũng bao gồm một lớp lấy trung bình, có thể là `mean`, `sum` hoặc `max`.

Dưới đây là mạng đã được chỉnh sửa để sử dụng `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Để chuẩn bị bộ dữ liệu cho việc huấn luyện, chúng ta cần cung cấp một hàm chuyển đổi sẽ chuẩn bị vector offset:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Lưu ý, khác với tất cả các ví dụ trước đây, mạng của chúng ta hiện nay chấp nhận hai tham số: vector dữ liệu và vector offset, có kích thước khác nhau. Tương tự, bộ tải dữ liệu của chúng ta cũng cung cấp cho ta 3 giá trị thay vì 2: cả vector văn bản và vector offset đều được cung cấp như các đặc trưng. Do đó, chúng ta cần điều chỉnh nhẹ hàm huấn luyện của mình để xử lý điều này:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Nhúng ngữ nghĩa: Word2Vec

Trong ví dụ trước của chúng ta, lớp nhúng mô hình học để ánh xạ từ thành biểu diễn vector, tuy nhiên biểu diễn này không mang nhiều ý nghĩa ngữ nghĩa. Sẽ rất tốt nếu học được biểu diễn vector sao cho các từ tương tự hoặc từ đồng nghĩa sẽ tương ứng với các vector gần nhau về khoảng cách vector nào đó (vd. khoảng cách Euclid).

Để làm điều đó, chúng ta cần tiền huấn luyện mô hình nhúng trên một tập văn bản lớn theo một cách cụ thể. Một trong những cách đầu tiên để huấn luyện nhúng ngữ nghĩa được gọi là [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Nó dựa trên hai kiến trúc chính được dùng để tạo biểu diễn phân tán của các từ:

 - **Continuous bag-of-words** (CBoW) — trong kiến trúc này, ta huấn luyện mô hình để dự đoán một từ dựa vào bối cảnh xung quanh. Với ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$, mục tiêu của mô hình là dự đoán $W_0$ từ $(W_{-2},W_{-1},W_1,W_2)$.
 - **Continuous skip-gram** là ngược lại so với CBoW. Mô hình sử dụng cửa sổ ngữ cảnh xung quanh các từ để dự đoán từ hiện tại.

CBoW nhanh hơn, trong khi skip-gram chậm hơn nhưng làm tốt hơn trong việc đại diện cho các từ ít xuất hiện.

![Hình ảnh minh họa cả thuật toán CBoW và Skip-Gram chuyển từ thành vector.](../../../../../translated_images/vi/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Để thử nghiệm nhúng word2vec đã được tiền huấn luyện trên bộ dữ liệu Google News, chúng ta có thể sử dụng thư viện **gensim**. Dưới đây là các từ gần nghĩa nhất với 'neural'

> **Chú ý:** Khi bạn tạo vector từ lần đầu, việc tải chúng có thể mất một khoảng thời gian!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Chúng ta cũng có thể tính toán embedding vector từ từ, để sử dụng trong việc huấn luyện mô hình phân loại (chúng tôi chỉ hiển thị 20 thành phần đầu tiên của vector để dễ nhìn):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Điều tuyệt vời về nhúng ngữ nghĩa là bạn có thể thao tác mã hóa vectơ để thay đổi ngữ nghĩa. Ví dụ, chúng ta có thể yêu cầu tìm một từ, biểu diễn vectơ của nó càng gần với từ *king* và *woman* càng tốt, và càng xa với từ *man* càng tốt: 


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Cả CBoW và Skip-Grams đều là các embeddings “dự đoán”, nghĩa là chúng chỉ xem xét ngữ cảnh cục bộ. Word2Vec không tận dụng ngữ cảnh toàn cục. 

**FastText** phát triển dựa trên Word2Vec bằng cách học các biểu diễn vector cho từng từ và các n-gram ký tự có trong từng từ. Các giá trị của các biểu diễn sau đó được trung bình thành một vector duy nhất ở mỗi bước huấn luyện. Mặc dù điều này tăng thêm rất nhiều tính toán cho quá trình tiền huấn luyện nhưng nó cho phép embeddings từ mã hóa thông tin dưới từ. 

Một phương pháp khác, **GloVe**, tận dụng ý tưởng về ma trận đồng xuất hiện, sử dụng các phương pháp mạng nơ-ron để phân rã ma trận đồng xuất hiện thành các vector từ phi tuyến và biểu đạt hơn.

Bạn có thể thử với ví dụ bằng cách đổi embeddings sang FastText và GloVe, vì gensim hỗ trợ nhiều mô hình embedding từ khác nhau.


## Sử dụng Embedding Được Huấn Luyện Sẵn trong PyTorch

Chúng ta có thể sửa đổi ví dụ trên để tiền điền ma trận trong lớp embedding của mình bằng các embedding ngữ nghĩa, chẳng hạn như Word2Vec. Chúng ta cần lưu ý rằng từ vựng của embedding được huấn luyện sẵn và kho văn bản của chúng ta có thể sẽ không khớp, vì vậy chúng ta sẽ khởi tạo trọng số cho các từ bị thiếu bằng các giá trị ngẫu nhiên:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Bây giờ chúng ta hãy huấn luyện mô hình của mình. Lưu ý rằng thời gian để huấn luyện mô hình lớn hơn đáng kể so với ví dụ trước, do kích thước lớp nhúng lớn hơn, và do đó số lượng tham số cũng cao hơn nhiều. Ngoài ra, vì điều này, chúng ta có thể cần huấn luyện mô hình trên nhiều ví dụ hơn nếu muốn tránh bị overfitting.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

Trong trường hợp của chúng tôi, chúng tôi không thấy sự tăng đáng kể về độ chính xác, điều này có thể do từ vựng khá khác nhau. 
Để khắc phục vấn đề về từ vựng khác nhau, chúng ta có thể sử dụng một trong các giải pháp sau:
* Huấn luyện lại mô hình word2vec trên từ vựng của chúng ta
* Tải bộ dữ liệu của chúng ta với từ vựng từ mô hình word2vec đã được huấn luyện trước. Từ vựng dùng để tải bộ dữ liệu có thể được chỉ định trong quá trình tải.

Cách tiếp cận sau có vẻ dễ dàng hơn, đặc biệt vì framework PyTorch `torchtext` có hỗ trợ nhúng sẵn. Ví dụ, chúng ta có thể khởi tạo từ vựng dựa trên GloVe theo cách sau:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Từ điển đã tải có các phép toán cơ bản sau:
* `vocab.stoi` cho phép chúng ta chuyển đổi từ thành chỉ số trong từ điển của nó
* `vocab.itos` làm ngược lại - chuyển đổi số thành từ
* `vocab.vectors` là mảng các vector embedding, do đó để lấy embedding của một từ `s` chúng ta cần sử dụng `vocab.vectors[vocab.stoi[s]]`

Dưới đây là ví dụ về thao tác với các embedding để minh họa phương trình **kind-man+woman = queen** (Tôi đã phải điều chỉnh hệ số một chút để nó hoạt động):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Để huấn luyện bộ phân loại bằng cách sử dụng những embeddings đó, trước tiên chúng ta cần mã hóa bộ dữ liệu của mình bằng từ vựng GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Như chúng ta đã thấy ở trên, tất cả các vector nhúng đều được lưu trữ trong ma trận `vocab.vectors`. Điều này làm cho việc tải các trọng số đó vào trọng số của lớp embedding trở nên rất dễ dàng bằng cách sao chép đơn giản:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Bây giờ hãy cùng huấn luyện mô hình của chúng ta và xem liệu chúng ta có nhận được kết quả tốt hơn không:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Một trong những lý do khiến chúng ta không thấy sự tăng đáng kể về độ chính xác là do một số từ trong bộ dữ liệu của chúng ta bị thiếu trong từ vựng GloVe đã được huấn luyện trước, và do đó chúng về cơ bản bị bỏ qua. Để khắc phục thực tế này, chúng ta có thể tự đào tạo các embeddings trên bộ dữ liệu của mình.


## Biểu diễn nhúng ngữ cảnh

Một hạn chế chính của các biểu diễn nhúng đào tạo trước truyền thống như Word2Vec là vấn đề phân biệt nghĩa từ. Mặc dù các biểu diễn nhúng đào tạo trước có thể nắm bắt một phần nghĩa của từ trong ngữ cảnh, nhưng mọi nghĩa có thể có của một từ đều được mã hóa vào cùng một biểu diễn nhúng. Điều này có thể gây ra vấn đề trong các mô hình phía sau, bởi vì nhiều từ như từ 'play' có nhiều nghĩa khác nhau tùy theo ngữ cảnh mà chúng được sử dụng.

Ví dụ, từ 'play' trong hai câu khác nhau dưới đây có nghĩa khá khác nhau:
- Tôi đã đi xem một **vở kịch** ở rạp hát.
- John muốn **chơi** với bạn bè của mình.

Các biểu diễn nhúng đào tạo trước ở trên đại diện cho cả hai nghĩa này của từ 'play' trong cùng một biểu diễn nhúng. Để vượt qua hạn chế này, chúng ta cần xây dựng biểu diễn nhúng dựa trên **mô hình ngôn ngữ**, được đào tạo trên một tập hợp văn bản lớn, và *biết* cách các từ có thể được ghép lại trong các ngữ cảnh khác nhau. Việc thảo luận về biểu diễn nhúng ngữ cảnh nằm ngoài phạm vi của hướng dẫn này, nhưng chúng ta sẽ trở lại với chúng khi nói về các mô hình ngôn ngữ trong đơn vị tiếp theo.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
